# Per-Residue Structural Variance (PRSV)

This Colab notebook runs `per-residue-structural-variance.R` from the `cianfrocco-lab/nvangos` repository. It installs the R dependencies, loads the PRSV function, reads aligned tubulin PDB files, and writes PRSV outputs.

Use the interactive widget panel to choose the alpha- and beta-tubulin chain IDs separately for the base PDB and comparison PDB.

## 1. Clone the repository

In [ ]:
!git clone --depth 1 https://github.com/cianfrocco-lab/nvangos.git
%cd nvangos

## 2. Install R dependencies

In [ ]:
%%bash
Rscript - <<'RSCRIPT'
packages <- c("tidyverse", "bio3d", "ggpubr")
installed <- rownames(installed.packages())
missing <- setdiff(packages, installed)
if (length(missing) > 0) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}
RSCRIPT

## 3. Provide input PDB files

Option A uploads PDB files from your computer. Option B downloads PDB files from URLs. Run one option, then use the widget panel below.

In [ ]:
# Option A: upload local PDB files.
from google.colab import files

uploaded = files.upload()
list(uploaded.keys())

In [ ]:
# Option B: download PDB files from URLs.
# Replace these example strings with direct download URLs, then run this cell.
import urllib.request

pdb_urls = {
    # "base_model.pdb": "https://example.org/base_model.pdb",
    # "comparison_model.pdb": "https://example.org/comparison_model.pdb",
}

for output_name, url in pdb_urls.items():
    urllib.request.urlretrieve(url, output_name)
    print(f"Downloaded {output_name}")

## 4. Choose files and chain IDs

In [ ]:
import glob
import os
import ipywidgets as widgets
from IPython.display import display

def pdb_files():
    files = sorted(glob.glob("*.pdb"))
    if not files:
        raise FileNotFoundError("No .pdb files found. Upload or download PDB files first, then rerun this cell.")
    return files

def chain_ids(path):
    chains = []
    seen = set()
    with open(path) as handle:
        for line in handle:
            if line.startswith(("ATOM", "HETATM")):
                chain = line[21].strip() if len(line) > 21 else ""
                label = chain if chain else "(blank)"
                if label not in seen:
                    seen.add(label)
                    chains.append(label)
    if not chains:
        raise ValueError(f"No ATOM/HETATM chain IDs found in {path}")
    return chains

def label_to_chain(label):
    return "" if label == "(blank)" else label

files_available = pdb_files()

base_pdb = widgets.Dropdown(options=files_available, description="Base PDB:", style={"description_width": "initial"})
comparison_pdb = widgets.Dropdown(options=files_available, value=files_available[min(1, len(files_available) - 1)], description="Comparison PDB:", style={"description_width": "initial"})

base_alpha_chain = widgets.Dropdown(description="Base alpha chain:", style={"description_width": "initial"})
base_beta_chain = widgets.Dropdown(description="Base beta chain:", style={"description_width": "initial"})
comparison_alpha_chain = widgets.Dropdown(description="Comparison alpha chain:", style={"description_width": "initial"})
comparison_beta_chain = widgets.Dropdown(description="Comparison beta chain:", style={"description_width": "initial"})
comparison_name = widgets.Text(value="base-vs-comparison", description="Output prefix:", style={"description_width": "initial"})

def set_default_chain(dropdown, options, preferred):
    dropdown.options = options
    dropdown.value = preferred if preferred in options else options[0]

def update_base_chains(*args):
    options = chain_ids(base_pdb.value)
    set_default_chain(base_alpha_chain, options, "A")
    set_default_chain(base_beta_chain, options, "B")

def update_comparison_chains(*args):
    options = chain_ids(comparison_pdb.value)
    set_default_chain(comparison_alpha_chain, options, "A")
    set_default_chain(comparison_beta_chain, options, "B")

base_pdb.observe(update_base_chains, names="value")
comparison_pdb.observe(update_comparison_chains, names="value")
update_base_chains()
update_comparison_chains()

display(widgets.VBox([
    widgets.HBox([base_pdb, comparison_pdb]),
    widgets.HBox([base_alpha_chain, base_beta_chain]),
    widgets.HBox([comparison_alpha_chain, comparison_beta_chain]),
    comparison_name,
]))

## 5. Run `per-residue-structural-variance.R`

In [ ]:
import os
import subprocess
import textwrap

BASE_PDB = base_pdb.value
COMPARISON_PDB = comparison_pdb.value
BASE_ALPHA_CHAIN = label_to_chain(base_alpha_chain.value)
BASE_BETA_CHAIN = label_to_chain(base_beta_chain.value)
COMPARISON_ALPHA_CHAIN = label_to_chain(comparison_alpha_chain.value)
COMPARISON_BETA_CHAIN = label_to_chain(comparison_beta_chain.value)
COMPARISON_NAME = comparison_name.value.strip() or "prsv-comparison"

OUTPUT_CSV = f"{COMPARISON_NAME}_prsv.csv"
OUTPUT_PNG = f"{COMPARISON_NAME}_prsv.png"

runner = r'''
source("per-residue-structural-variance.R")

base_path <- Sys.getenv("BASE_PDB")
comparison_path <- Sys.getenv("COMPARISON_PDB")
comparison_name <- Sys.getenv("COMPARISON_NAME")
output_csv <- Sys.getenv("OUTPUT_CSV")
output_png <- Sys.getenv("OUTPUT_PNG")

base <- bio3d::read.pdb(base_path)
comparison <- bio3d::read.pdb(comparison_path)

result <- prsv(base, comparison,
               base.alpha.chain = Sys.getenv("BASE_ALPHA_CHAIN"),
               base.beta.chain = Sys.getenv("BASE_BETA_CHAIN"),
               comp.alpha.chain = Sys.getenv("COMPARISON_ALPHA_CHAIN"),
               comp.beta.chain = Sys.getenv("COMPARISON_BETA_CHAIN")) %>%
  mutate(Comparison = comparison_name)
write.csv(result, output_csv, row.names = FALSE)

plot <- result %>%
  ggplot(aes(x = Res, y = PRSV, color = Tubulin)) +
  geom_line(linewidth = 0.6) +
  facet_wrap(~Tubulin, ncol = 1, scales = "free_x") +
  labs(x = "Residue", y = expression(bold(paste("Per Residue Structural Variance ", (ring(A)^2)))), title = comparison_name) +
  theme_pubr() +
  theme(plot.title = element_text(face = "bold"))

ggsave(output_png, plot, width = 7, height = 5, dpi = 300)
print(head(result))
message("Wrote ", output_csv)
message("Wrote ", output_png)
'''

with open("run_prsv_colab.R", "w") as handle:
    handle.write(textwrap.dedent(runner))

env = os.environ.copy()
env.update({
    "BASE_PDB": BASE_PDB,
    "COMPARISON_PDB": COMPARISON_PDB,
    "BASE_ALPHA_CHAIN": BASE_ALPHA_CHAIN,
    "BASE_BETA_CHAIN": BASE_BETA_CHAIN,
    "COMPARISON_ALPHA_CHAIN": COMPARISON_ALPHA_CHAIN,
    "COMPARISON_BETA_CHAIN": COMPARISON_BETA_CHAIN,
    "COMPARISON_NAME": COMPARISON_NAME,
    "OUTPUT_CSV": OUTPUT_CSV,
    "OUTPUT_PNG": OUTPUT_PNG,
})

subprocess.run(["Rscript", "run_prsv_colab.R"], check=True, env=env)

## 6. Preview and download outputs

In [ ]:
import pandas as pd
from IPython.display import Image, display
from google.colab import files

display(pd.read_csv(OUTPUT_CSV).head())
display(Image(filename=OUTPUT_PNG))

files.download(OUTPUT_CSV)
files.download(OUTPUT_PNG)